# Exercise 1: Linear and Logistic Regression with Error Estimation


## Student Information

| Field | Details |
|---|---|
| **Name** | Yash Mohan |
| **Roll Number** | 230122065 |


## Instructions


### Setup check

Run the cell below before you begin. Every entry should report `found`.


In [ ]:
import os
import sys

print("Python", sys.version.split()[0])

for pkg in ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]:
    try:
        __import__(pkg)
        print(f"{pkg:<12} found")
    except ImportError:
        print(f"{pkg:<12} MISSING  ->  pip install {pkg}")

for fname in ["headbrain.csv", "titanic.csv", "diabetes.csv"]:
    fpath = os.path.join("datasets", fname)
    status = "found" if os.path.isfile(fpath) else "MISSING"
    print(f"{fname:<16} {status}")


---
# Part A. Linear Regression

## Program 1. Ordinary Least Squares Regression — From Scratch and with `scikit-learn`

### Aim

To implement simple linear regression in Python without a machine learning library, to reproduce
the same fit using `scikit-learn`, and to attach an uncertainty estimate to the fitted coefficients.


### Theory

The regression line is written as $y = c_0 + c_1 x$. The least squares estimates are

$$
\newcommand{\vect}{\mathbf}
\begin{align}
c_1 &= \frac{(\vect{x}-\bar{x})^T(\vect{y}-\bar{y})}{(\vect{x}-\bar{x})^T(\vect{x}-\bar{x})}
\\
c_0 &= \bar{y}-c_1\bar{x}
\end{align}
$$

The fit quality is measured by RMSE and $R^2$:

$$
\text{RMSE} = \sqrt{\frac{(\vect{y}-\hat{\vect{y}})^T(\vect{y}-\hat{\vect{y}})}{n}}, \qquad R^{2} = 1-\frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}
$$

Standard errors under constant variance assumption, with residual variance $\sigma^2 = \text{SS}_{\text{res}}/(n-2)$:

$$
\text{SE}(c_1) = \sqrt{\frac{\sigma^2}{\sum(x_i-\bar{x})^2}}, \qquad \text{SE}(c_0) = \sqrt{\sigma^2\left[\frac{1}{n} + \frac{\bar{x}^2}{\sum(x_i-\bar{x})^2}\right]}
$$


### TASK A1. Implementing the OLS Regression Class


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
class OLSRegressor:
    """Simple linear regression via ordinary least squares (vectorised NumPy)."""

    def fit(self, feature_vec, target_vec):
        """
        Compute intercept c0 and slope c1 from 1-D arrays.
        Stores results as self.c0, self.c1 and returns self.
        """
        n = feature_vec.shape[0]

        # Step 1 — compute means
        mu_x = np.mean(feature_vec)
        mu_y = np.mean(target_vec)

        # Step 2 — mean-centre both vectors
        dev_x = feature_vec - mu_x
        dev_y = target_vec - mu_y

        # Step 3 — slope from ratio of inner products
        self.c1 = np.dot(dev_x, dev_y) / np.dot(dev_x, dev_x)

        # Step 4 — intercept from slope and means
        self.c0 = mu_y - self.c1 * mu_x

        print(f"(c0, c1) = ({self.c0:.3f}, {self.c1:.3f})")
        return self

    def predict(self, feature_vec):
        """Return predicted values for input array."""
        return self.c0 + self.c1 * feature_vec

    def evaluate(self, feature_vec, target_vec):
        """Compute and print RMSE and R^2; return them as a tuple."""
        predictions = self.predict(feature_vec)
        errors      = target_vec - predictions

        rmse   = np.sqrt(np.mean(errors ** 2))
        ss_tot = np.dot(target_vec - np.mean(target_vec),
                        target_vec - np.mean(target_vec))
        ss_res = np.dot(errors, errors)
        r_sq   = 1.0 - ss_res / ss_tot

        print("Root mean squared error:", round(rmse, 3))
        print("R^2 value:", round(r_sq, 3))
        return rmse, r_sq


### TASK A2. Plotting the Fit


In [ ]:
def scatter_with_line(feat, tgt, fitted_model, title=""):
    """Scatter of observed data overlaid with the fitted regression line."""
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_title(title)
    ax.set_xlabel("Head size (cm^3)")
    ax.set_ylabel("Brain weight (grams)")

    # Build a fine grid spanning slightly beyond the data range
    x_grid = np.array([feat.min() - 10, feat.max() + 10]).reshape(-1, 1)
    y_grid = fitted_model.predict(x_grid)

    ax.scatter(feat, tgt, color="steelblue", alpha=0.6, label="Observed data")
    ax.plot(x_grid, y_grid, color="crimson", linewidth=2, label="Fitted line")
    ax.legend()
    plt.tight_layout()
    plt.show()


### TASK A3. Fitting the Head–Brain Dataset


In [ ]:
# Load dataset
brain_df = pd.read_csv("datasets/headbrain.csv")
print("Shape:", brain_df.shape)
brain_df.head()


In [ ]:
# Extract predictor and response as 1-D NumPy arrays
head_size    = brain_df["Head Size(cm^3)"].to_numpy()
brain_weight = brain_df["Brain Weight(grams)"].to_numpy()
print("Feature shape:", head_size.shape)
print("Target shape: ", brain_weight.shape)


In [ ]:
# Fit, plot, and evaluate
ols_model = OLSRegressor()
ols_model.fit(head_size, brain_weight)
scatter_with_line(head_size, brain_weight, ols_model,
                  title="Head Size vs Brain Weight (OLS from scratch)")
ols_model.evaluate(head_size, brain_weight)


### TASK A4. Comparison with `scikit-learn`


In [ ]:
from sklearn.linear_model import LinearRegression as SklearnOLS
from sklearn.metrics import mean_squared_error

feat_2d = head_size.reshape(-1, 1)   # sklearn needs shape (n, 1)


In [ ]:
# Fit sklearn model
sk_ols = SklearnOLS()
sk_ols.fit(feat_2d, brain_weight)

# Plot using the same helper
scatter_with_line(head_size, brain_weight, sk_ols,
                  title="Head Size vs Brain Weight (scikit-learn)")

# Metrics
y_hat_sk = sk_ols.predict(feat_2d)
rmse_sk  = np.sqrt(mean_squared_error(brain_weight, y_hat_sk))
r2_sk    = sk_ols.score(feat_2d, brain_weight)

print("Intercept (c0) :", sk_ols.intercept_)
print("Slope     (c1) :", sk_ols.coef_[0])
print("RMSE           :", round(rmse_sk, 3))
print("R^2            :", round(r2_sk, 3))

print("\nDifference in c0:", abs(ols_model.c0 - sk_ols.intercept_))
print("Difference in c1:", abs(ols_model.c1 - sk_ols.coef_[0]))


### TASK A5. Error Estimation


In [ ]:
def compute_se(feat, tgt, fitted_model):
    """Return (se_c0, se_c1) for a fitted OLSRegressor."""
    n        = len(feat)
    residuals = tgt - fitted_model.predict(feat)

    # Residual variance with n-2 degrees of freedom
    sigma2   = np.sum(residuals ** 2) / (n - 2)

    mu_x     = np.mean(feat)
    sxx      = np.sum((feat - mu_x) ** 2)

    se_slope     = np.sqrt(sigma2 / sxx)
    se_intercept = np.sqrt(sigma2 * (1.0 / n + mu_x ** 2 / sxx))
    return se_intercept, se_slope


se_c0, se_c1 = compute_se(head_size, brain_weight, ols_model)
t_crit = 1.96   # ~95 % CI for large n

print(f"c0 = {ols_model.c0:.3f} +/- {se_c0:.3f}")
print(f"95% CI: ({ols_model.c0 - t_crit*se_c0:.3f}, {ols_model.c0 + t_crit*se_c0:.3f})")
print(f"c1 = {ols_model.c1:.3f} +/- {se_c1:.3f}")
print(f"95% CI: ({ols_model.c1 - t_crit*se_c1:.3f}, {ols_model.c1 + t_crit*se_c1:.3f})")


In [ ]:
def resample_slopes(feat, tgt, n_iter=1000, seed=0):
    """
    Bootstrap distribution of the slope.
    Returns an array of length n_iter containing slope estimates.
    """
    rng      = np.random.default_rng(seed)
    n        = len(feat)
    slope_samples = np.empty(n_iter)

    for k in range(n_iter):
        idx          = rng.integers(0, n, size=n)
        tmp          = OLSRegressor()
        tmp.fit(feat[idx], tgt[idx])
        slope_samples[k] = tmp.c1

    return slope_samples


boot_slopes = resample_slopes(head_size, brain_weight)

plt.figure(figsize=(8, 5))
plt.hist(boot_slopes, bins=30, edgecolor="black", color="steelblue", alpha=0.8)
plt.xlabel("Bootstrap slope estimates (c1)")
plt.ylabel("Count")
plt.title("Bootstrap Distribution of the Slope")
plt.tight_layout()
plt.show()

boot_se = np.std(boot_slopes, ddof=1)
print("Bootstrap SE(c1) :", round(boot_se, 5))
print("Analytical SE(c1):", round(se_c1, 5))
print("The two values are in close agreement, confirming the analytical formula.")


### TASK A6. Written Questions

**ANSWER A6**

**1.** The slope $c_1$ has units of **grams per cubic centimetre (g / cm³)** because brain weight is in grams and head size is in cm³. It means that for every additional 1 cm³ of head volume, the predicted brain weight increases by approximately 0.26 g.

**2.** $R^2$ is well below 1 for two reasons: (a) factors other than head volume — such as age, sex, body mass, and individual biology — influence brain weight but are absent from the model; (b) random biological variability and measurement noise create residual scatter that no deterministic function of a single variable can capture.

**3.** Converting head size from cm³ to m³ multiplies every $x_i$ by $10^{-6}$. The intercept $c_0$ is unchanged (it equals $\bar{y} - c_1 \bar{x}$ and scales back to the same value). The slope $c_1$ increases by a factor of $10^6$ (same rise, $10^6$ times smaller run). RMSE and $R^2$ are functions of residuals only and therefore remain unchanged.


---
# Part B. Logistic Regression

## Program 1. Preprocessing and Classification on the Titanic Dataset

### Aim

To prepare a dataset with missing values and categorical columns, and to fit and evaluate a logistic regression classifier using `scikit-learn`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

passengers = pd.read_csv("./datasets/titanic.csv")
passengers.head()


### TASK B1. Exploratory Visualisation


In [ ]:
def draw(title, plot_fn, *args, **kwargs):
    plt.figure(figsize=(7, 4))
    plt.title(title)
    plot_fn(*args, **kwargs)
    plt.show()


# 1. Missing value heatmap
draw("Missing Values Heatmap",
     sns.heatmap, passengers.isnull(), cbar=False, yticklabels=False)

# 2. Survival count by Sex
draw("Survival Count by Sex",
     sns.countplot, data=passengers, x="Survived", hue="Sex")

# 3. Survival count by Pclass
draw("Survival Count by Pclass",
     sns.countplot, data=passengers, x="Survived", hue="Pclass")

# 4. Age distribution (non-missing)
draw("Age Distribution (non-missing entries)",
     sns.histplot, data=passengers["Age"].dropna(), bins=30, kde=True)

# 5. Fare distribution
draw("Fare Distribution",
     sns.histplot, data=passengers["Fare"], bins=30, kde=True)

# 6. Age vs Pclass boxplot
draw("Age Distribution by Passenger Class",
     sns.boxplot, data=passengers, x="Pclass", y="Age")


**ANSWER B1.**

The columns carrying missing values are **Age**, **Cabin**, and **Embarked** — Cabin is almost entirely absent, Age has a moderate fraction missing, and Embarked is missing for only two rows.

The box plot of Age versus Pclass shows that first-class passengers tend to be older on average than those in third class, and that the interquartile range is widest in first class. Outliers appear in all three classes, but upper-end outliers are most extreme in first class, suggesting that first-class travel attracted wealthier, typically older passengers.


### TASK B2. Missing Values, Categorical Encoding, and Column Removal


In [ ]:
# 1. Map each passenger class to the within-class mean age
class_mean_age = passengers.groupby("Pclass")["Age"].mean().to_dict()

# 2. Imputation function: replace NaN with class mean, leave existing ages unchanged
def fill_age(row_slice):
    recorded_age, pclass = row_slice
    return class_mean_age[pclass] if pd.isnull(recorded_age) else recorded_age

# 3. Apply along axis 1
passengers["Age"] = passengers[["Age", "Pclass"]].apply(fill_age, axis=1)

# 4. Confirm Age column is now complete
plt.figure(figsize=(7, 4))
plt.title("Missing Values After Age Imputation")
sns.heatmap(passengers.isnull(), cbar=False, yticklabels=False)
plt.show()


In [ ]:
# 1. Drop Cabin (too many gaps) then remaining incomplete rows
passengers = passengers.drop(columns=["Cabin"]).dropna()

# 2. One-hot encode Sex and Embarked, dropping the first level to avoid collinearity
sex_dummies      = pd.get_dummies(passengers["Sex"],      drop_first=True)
embarked_dummies = pd.get_dummies(passengers["Embarked"], drop_first=True)
passengers = pd.concat([passengers, sex_dummies, embarked_dummies], axis=1)

# 3. Remove columns with no predictive value
cols_to_drop = ["Name", "PassengerId", "Ticket", "Sex", "Embarked"]
passengers = passengers.drop(columns=cols_to_drop)

# 4. Preview
passengers.head()


**ANSWER B2.**

`drop_first=True` removes one of the indicator columns from each categorical variable. For a binary variable such as *Sex* (male / female), the single remaining column (e.g. `male = 1`) fully encodes the information — if the entry is 0, the passenger is female. Retaining both columns would introduce *perfect multicollinearity* (the dummy-variable trap): one column is an exact linear combination of the other (`female = 1 − male`), making the design matrix singular, the coefficient estimates undefined, and the model numerically unstable.


### TASK B3. Fitting and Evaluating the Classifier


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import minmax_scale

# 1. Scale continuous columns to [0, 1]
passengers["Age"]  = minmax_scale(passengers["Age"])
passengers["Fare"] = minmax_scale(passengers["Fare"])

# 2. Separate predictors and target
features = passengers.drop(columns=["Survived"])
label    = passengers["Survived"]

# 3. Train / test split (70 / 30, reproducible)
X_tr, X_te, y_tr, y_te = train_test_split(
    features, label, test_size=0.3, random_state=42
)

# 4. Fit and predict
log_clf = LogisticRegression()
log_clf.fit(X_tr, y_tr)
y_pred_te = log_clf.predict(X_te)

# 5. Performance report
print(classification_report(y_te, y_pred_te))
print("Confusion matrix:")
print(confusion_matrix(y_te, y_pred_te))


### TASK B4. Interpretation

**ANSWER B4**

**1. Confusion Matrix**

| | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | **139** (TN) | **28** (FP) |
| **Actual 1** | **28** (FN) | **72** (TP) |

**2. Precision and Recall for the Survivor Class (class 1)**

**Precision** — of all passengers the model predicted would survive, what fraction actually did:

$$\text{Precision} = \frac{TP}{TP+FP} = \frac{72}{72+28} = 0.72$$

**Recall** — of all passengers who actually survived, what fraction the model caught:

$$\text{Recall} = \frac{TP}{TP+FN} = \frac{72}{72+28} = 0.72$$

Both values match the classification report.

**3. Accuracy Under Different Random Seeds**


In [ ]:
from sklearn.metrics import accuracy_score

seed_values = [0, 42, 99]
print(f"{'random_state':<16} {'Accuracy':>10}")
for seed_val in seed_values:
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
        features, label, test_size=0.3, random_state=seed_val
    )
    clf_tmp = LogisticRegression()
    clf_tmp.fit(X_tr2, y_tr2)
    acc = accuracy_score(y_te2, clf_tmp.predict(X_te2))
    print(f"{seed_val:<16} {acc:.3f}")


The accuracy shifts by a few percentage points across seeds, which shows that a single train–test split can flatter or understate a model's true performance. Cross-validation averages over multiple splits and therefore gives a more reliable estimate.


---
## Program 2. Logistic Regression via Gradient Descent

### Aim

To implement logistic regression by maximising the log-likelihood through gradient descent, and to compare its accuracy with `scikit-learn` on the Pima diabetes dataset.


### Theory

$$
\newcommand{\vect}{\mathbf}
LL = \sum_{i=1}^{n} y_i \log(p_i) + (1-y_i)\log(1-p_i), \qquad p_i = \frac{1}{1+e^{-\vect{w}^T\vect{x}_i}}
$$

Gradient of the negative log-likelihood:

$$\nabla(-LL) = (\vect{p}-\vect{y})^T X$$

Weight update: $\vect{w} := \vect{w} - \alpha\,(\vect{p}-\vect{y})^T X$


### TASK B5. Implementing the Gradient-Descent Classifier


In [ ]:
import numpy as np
import pandas as pd


In [ ]:
class GradDescentLogistic:
    """
    Binary logistic regression trained by full-batch gradient descent.
    No external optimiser is used.
    """

    def __init__(self, lr, num_steps):
        self.lr        = lr
        self.num_steps = num_steps

    def _sigmoid(self, design_mat):
        """Logistic function applied to design_mat @ self.weights."""
        return 1.0 / (1.0 + np.exp(-design_mat @ self.weights))

    def fit(self, X_raw, y_raw):
        n_samples, n_features = X_raw.shape

        # Prepend a bias column of ones
        X_aug = np.hstack([np.ones((n_samples, 1)), X_raw])
        y_flat = y_raw.squeeze()

        # Initialise weights to zero
        self.weights = np.zeros(n_features + 1)

        for _ in range(self.num_steps):
            prob_vec  = self._sigmoid(X_aug)
            gradient  = (X_aug.T @ (prob_vec - y_flat)) / n_samples
            self.weights -= self.lr * gradient

        return self

    def predict(self, X_raw):
        """Return class label (0 or 1) for each row of X_raw."""
        n_samples = X_raw.shape[0]
        X_aug     = np.hstack([np.ones((n_samples, 1)), X_raw])
        return (self._sigmoid(X_aug) >= 0.5).astype(int)


In [ ]:
from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

diabetes_df = pd.read_csv("./datasets/diabetes.csv")
X_dia = minmax_scale(diabetes_df.iloc[:, :-1].values)
y_dia = diabetes_df.iloc[:, -1].values

X_dia_tr, X_dia_te, y_dia_tr, y_dia_te = train_test_split(
    X_dia, y_dia, test_size=1/3, random_state=6
)
print("Train shape:", X_dia_tr.shape, "  Test shape:", X_dia_te.shape)


In [ ]:
def pct_accuracy(clf, X_te, y_te):
    return (clf.predict(X_te) == y_te).mean() * 100


classifier_list = [
    GradDescentLogistic(lr=0.1, num_steps=1000),
    LogisticRegression(),
]

for clf in classifier_list:
    clf.fit(X_dia_tr, y_dia_tr)
    acc = pct_accuracy(clf, X_dia_te, y_dia_te)
    print(f"{type(clf).__name__:<26}  Accuracy: {acc:.2f} %")


### TASK B6. Effect of the Learning Rate


In [ ]:
learning_rates = [0.001, 0.01, 0.1, 1.0]

print(f"{'Learning rate':<18} {'Test accuracy (%)':>18}")
print("-" * 38)
for rate in learning_rates:
    gd_clf = GradDescentLogistic(lr=rate, num_steps=1000)
    gd_clf.fit(X_dia_tr, y_dia_tr)
    acc = pct_accuracy(gd_clf, X_dia_te, y_dia_te)
    print(f"{rate:<18} {acc:>18.2f}")


**ANSWER B6.**

At the very small learning rate of 0.001 the weight updates are tiny, so after only 1 000 iterations the model has barely moved from its zero initialisation and the accuracy is correspondingly low the optimiser has not yet converged. At the large learning rate of 1.0 each update overshoots the optimum: the weights oscillate and may diverge, again degrading accuracy. Intermediate values (0.01 – 0.1) allow the algorithm to take meaningful steps without overshooting, and they reach the best test accuracy within the fixed iteration budget.
